# Module 04: Interactive Object-Oriented Architecture, Memory & MRO

Welcome to the deep interactive laboratory for **Module 04: Deep OOP**!
In this lab, you will explore the physical mechanics and runtime behavior of Python objects:
1. **The Object Lifecycle:** Why `__new__` allocates memory before `__init__` initializes attributes.
2. **Instance vs Class Namespace:** Tracing attribute resolution in `__dict__`.
3. **Encapsulation & Protection:** Property descriptors (`@property`, `@setter`) and invariant validation.
4. **C3 Linearization & Diamond Inheritance:** How Python calculates Method Resolution Order (MRO).
5. **Dunder Protocols & Operator Overloading:** Custom arithmetic, comparison, and string formatting.
6. **Abstract Base Classes (ABCs):** Formal interface enforcement with `abc.ABC`.
7. **Interactive Challenge:** Fixing a broken multi-tier plugin hierarchy.


## 1. The Allocation & Initialization Lifecycle (`__new__` vs `__init__`)


In [ ]:
class MemoryTracedObject:
    def __new__(cls, *args, **kwargs):
        print(f"[1. ALLOCATION] __new__ called for class {cls.__name__}")
        # __new__ allocates the raw block of memory and returns the uninitialized instance
        instance = super().__new__(cls)
        print(f"    Raw memory allocated at id: {hex(id(instance))}")
        return instance

    def __init__(self, name: str):
        print(f"[2. INITIALIZATION] __init__ called for instance {hex(id(self))}")
        self.name = name
        print(f"    Instance state initialized: name = {self.name!r}")

obj = MemoryTracedObject("ProductionServer")
print(f"Final Object: {obj}, Type: {type(obj)}")


## 2. Inspecting Namespaces: `__dict__` and Attribute Lookup


In [ ]:
class ServerNode:
    CLUSTER_REGION = "us-east-1"  # Class attribute: lives in ServerNode.__dict__

    def __init__(self, node_id: str, ip: str):
        self.node_id = node_id     # Instance attribute: lives in self.__dict__
        self.ip = ip

node1 = ServerNode("node-01", "10.0.0.1")
node2 = ServerNode("node-02", "10.0.0.2")

print("node1.__dict__:", node1.__dict__)
print("ServerNode class attributes:", [k for k in ServerNode.__dict__ if not k.startswith('__')])

# Demonstrating lookup resolution:
print("node1.CLUSTER_REGION:", node1.CLUSTER_REGION)

# Overriding on an instance creates an entry in that instance's dict without affecting the class!
node1.CLUSTER_REGION = "eu-central-1"
print("After override:")
print("node1.__dict__:", node1.__dict__)
print("node2.CLUSTER_REGION (unaffected):", node2.CLUSTER_REGION)


## 3. Pythonic Encapsulation with `@property`


In [ ]:
class BankAccount:
    def __init__(self, owner: str, initial_balance: float = 0.0):
        self.owner = owner
        self._balance = initial_balance

    @property
    def balance(self) -> float:
        """Read-only view of balance."""
        return self._balance

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError(f"Deposit amount must be positive, got {amount}")
        self._balance += amount

    def withdraw(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        if amount > self._balance:
            raise ValueError(f"Insufficient funds! Requested: ${amount:.2f}, Available: ${self._balance:.2f}")
        self._balance -= amount

account = BankAccount("Alice", 250.0)
print(f"Initial balance: ${account.balance:.2f}")
account.deposit(100.0)
print(f"After deposit: ${account.balance:.2f}")

try:
    account.withdraw(500.0)
except ValueError as err:
    print(f"[EXPECTED REJECTION]: {err}")


## 4. Diamond Inheritance & C3 Linearization MRO


In [ ]:
# The classic Diamond Problem:
#       A
#      / \
#     B   C
#      \ /
#       D

class A:
    def ping(self): return "A"

class B(A):
    def ping(self): return f"B -> {super().ping()}"

class C(A):
    def ping(self): return f"C -> {super().ping()}"

class D(B, C):
    def ping(self): return f"D -> {super().ping()}"

d_instance = D()
print("Execution call chain:", d_instance.ping())
print("\nMethod Resolution Order (MRO):")
for idx, cls in enumerate(D.mro()):
    print(f"  [{idx}] {cls.__name__}")


## 5. Dunder Magic Methods & Operator Overloading


In [ ]:
from functools import total_ordering


@total_ordering
class Currency:
    """Value Object representing money with strict currency checking and operator math."""

    def __init__(self, amount: float, code: str = "USD"):
        self.amount = round(float(amount), 2)
        self.code = code.upper()

    def __repr__(self) -> str:
        return f"Currency({self.amount:.2f}, {self.code!r})"

    def __str__(self) -> str:
        symbol = "$" if self.code == "USD" else self.code + " "
        return f"{symbol}{self.amount:.2f}"

    def __add__(self, other: 'Currency') -> 'Currency':
        if not isinstance(other, Currency) or self.code != other.code:
            raise TypeError(f"Cannot add {type(other).__name__} ({getattr(other, 'code', 'N/A')}) to {self.code}")
        return Currency(self.amount + other.amount, self.code)

    def __sub__(self, other: 'Currency') -> 'Currency':
        if not isinstance(other, Currency) or self.code != other.code:
            raise TypeError("Currency codes must match for subtraction")
        return Currency(self.amount - other.amount, self.code)

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Currency):
            return NotImplemented
        return (self.amount, self.code) == (other.amount, other.code)

    def __lt__(self, other: 'Currency') -> bool:
        if not isinstance(other, Currency) or self.code != other.code:
            raise TypeError("Cannot compare currencies with different codes")
        return self.amount < other.amount

c1 = Currency(120.50, "USD")
c2 = Currency(79.50, "USD")

print("c1:", c1)
print("c2:", c2)
print("c1 + c2:", c1 + c2)
print("c1 > c2:", c1 > c2)
assert (c1 + c2) == Currency(200.00, "USD")
print("[VERIFIED] Operator overloading assertions passed!")


## 6. Abstract Base Classes (ABCs) and Interface Contracts


In [ ]:
from abc import ABC, abstractmethod


class StorageBackend(ABC):
    """Abstract interface defining contract for storage adapters."""

    @abstractmethod
    def read(self, key: str) -> bytes:
        """Retrieve data associated with key."""
        pass

    @abstractmethod
    def write(self, key: str, payload: bytes) -> bool:
        """Persist data under key."""
        pass

# Attempting to instantiate an incomplete implementation fails immediately:
class IncompleteStorage(StorageBackend):
    def read(self, key: str) -> bytes:
        return b"data"

try:
    s = IncompleteStorage()
except TypeError as err:
    print(f"[EXPECTED ENFORCEMENT]: {err}")

# Complete implementation:
class MemoryStorage(StorageBackend):
    def __init__(self):
        self._data = {}

    def read(self, key: str) -> bytes:
        return self._data.get(key, b"")

    def write(self, key: str, payload: bytes) -> bool:
        self._data[key] = payload
        return True

mem = MemoryStorage()
mem.write("k1", b"hello world")
print("Read back:", mem.read("k1").decode())


## 7. Interactive Challenge: The Account Settlement Pipeline


In [ ]:
# CHALLENGE: Implement an AccountSettler class that:
# 1. Inherits from ABC
# 2. Implements settle_accounts(debtor_acc, creditor_acc, amount: Currency)
# 3. Ensures atomicity: if debtor has insufficient funds, neither account balance is modified!

class AbstractSettler(ABC):
    @abstractmethod
    def settle(self, debtor: BankAccount, creditor: BankAccount, amount: float) -> bool:
        pass

class ProductionSettler(AbstractSettler):
    def settle(self, debtor: BankAccount, creditor: BankAccount, amount: float) -> bool:
        if debtor.balance < amount:
            return False
        debtor.withdraw(amount)
        creditor.deposit(amount)
        return True

acc_a = BankAccount("Alice", 100.0)
acc_b = BankAccount("Bob", 50.0)
settler = ProductionSettler()

success = settler.settle(acc_a, acc_b, 30.0)
assert success is True
assert acc_a.balance == 70.0
assert acc_b.balance == 80.0

# Failure case:
fail_res = settler.settle(acc_a, acc_b, 1000.0)
assert fail_res is False
assert acc_a.balance == 70.0
print("[ALL CHALLENGE CHECKS PASSED] Perfect atomic settlement!")
